### Imports e conexão

In [8]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
conn = sqlite3.connect("../data/processed/olist.db")

## ⚠️ Limitação do dataset

O Olist gera um `customer_id` único por pedido, impossibilitando
rastrear recompra pelo mesmo cliente. Essa é uma limitação conhecida
do dataset, documentada pelo próprio Olist.

**Pivô analítico:** analisaremos cohort e retenção de **vendedores**
(equivalentes a restaurantes no iFood), que têm IDs estáveis ao longo
do tempo e representam um problema de negócio igualmente relevante:
*como manter restaurantes ativos e engajados na plataforma?*

### Prepara dados de vendedores

In [9]:
query = """
SELECT
    oi.seller_id,
    strftime('%Y-%m', o.order_purchase_timestamp) AS mes_pedido,
    COUNT(DISTINCT o.order_id)                    AS pedidos,
    ROUND(SUM(oi.price + oi.freight_value), 2)    AS receita
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1, 2
"""
df = pd.read_sql(query, conn)

df["cohort"]     = df.groupby("seller_id")["mes_pedido"].transform("min")
df["mes_pedido"] = pd.to_datetime(df["mes_pedido"])
df["cohort"]     = pd.to_datetime(df["cohort"])
df["periodo"]    = (
    (df["mes_pedido"].dt.year  - df["cohort"].dt.year) * 12 +
    (df["mes_pedido"].dt.month - df["cohort"].dt.month)
)

print(f"Vendedores únicos: {df['seller_id'].nunique():,}")
print(f"Período: {df['mes_pedido'].min().strftime('%Y-%m')} → {df['mes_pedido'].max().strftime('%Y-%m')}")
df.head()

Vendedores únicos: 2,970
Período: 2016-09 → 2018-08


,seller_id,mes_pedido,pedidos,receita,cohort,periodo
0,0015a82c2db000af6aaaf3ae2ecb0532,2017-09-01,1,916.02,2017-09-01,0
1,0015a82c2db000af6aaaf3ae2ecb0532,2017-10-01,2,1832.04,2017-09-01,1
2,001cca7ae9ae17fb1caed9dfb1094831,2017-02-01,5,1295.40,2017-02-01,0
3,001cca7ae9ae17fb1caed9dfb1094831,2017-03-01,11,2124.00,2017-02-01,1
4,001cca7ae9ae17fb1caed9dfb1094831,2017-04-01,13,2087.55,2017-02-01,2


### Monta tabela de cohort

In [10]:
cohort_data = (
    df.groupby(["cohort", "periodo"])["seller_id"]
    .nunique()
    .reset_index()
    .rename(columns={"seller_id": "vendedores"})
)

cohort_pivot = cohort_data.pivot_table(
    index="cohort", columns="periodo", values="vendedores"
)

cohort_size = cohort_pivot[0]

retention = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100
retention = retention.loc[
    cohort_size[cohort_size >= 20].index
].iloc[:, :13]

retention.index = retention.index.strftime("%Y-%m")
print(f"Cohorts analisados: {len(retention)}")
retention.head()

Cohorts analisados: 21


periodo,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort,,,,,,,,,,,,,
2016-10,100.0,NaN,NaN,54.3,64.6,59.1,55.1,58.3,51.2,51.2,50.4,47.2,49.6
2017-01,100.0,71.1,70.5,60.4,66.4,57.7,51.7,58.4,51.7,53.7,52.3,43.0,47.7
2017-02,100.0,63.1,56.1,57.9,46.7,51.9,51.9,45.8,44.4,42.5,45.8,38.8,38.3
2017-03,100.0,60.2,56.5,52.8,49.1,49.7,46.6,42.9,49.7,38.5,40.4,34.8,34.2
2017-04,100.0,55.0,44.1,51.4,47.7,45.9,37.8,45.0,38.7,38.7,31.5,35.1,31.5


### Heatmap de retenção de vendedores